# 01 — Chunking Hands-On

> **Goal of this notebook:** Understand chunking by doing it yourself — not just reading about it.
> You will load real text, apply different splitting strategies, inspect every chunk, and see exactly how boundaries affect what gets retrieved.

---

## What You Will Practice

1. ✅ Verify environment — confirm all libraries are installed
2. 📄 Understand the `Document` object — the core data structure in LangChain RAG
3. ✂️ Fixed-size chunking — simplest method, understand its weaknesses
4. 🔄 Recursive chunking — the smart default, understand why it is better
5. 📊 Compare both methods side by side on the same text
6. 🔢 Token counting — verify your chunks are within model context limits
7. 🏷️ Metadata — attach source information to every chunk

---

**Before running:** Make sure your VS Code kernel is set to `Handson/.venv/Scripts/python.exe`  
(`Ctrl+Shift+P` → `Notebook: Select Kernel` → pick `.venv`)

---
## Cell 1 — Environment Verification

Always run this first. It confirms every library is installed and tells you the version you are using.

In [1]:
# ── Environment Verification ──────────────────────────────────────────────────
# Run this first. If anything fails, your kernel is not pointing to .venv

import sys
print(f"Python version  : {sys.version}")
print(f"Python path     : {sys.executable}")
print()

# LangChain
import langchain
import langchain_community
print(f"langchain           : {langchain.__version__}")
print(f"langchain-community : {langchain_community.__version__}")

# Text splitters (moved to langchain_text_splitters in LangChain v1.x)
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter,
    MarkdownHeaderTextSplitter,
)
print("Text splitters      : ✅ all imported")

# Document object
from langchain_core.documents import Document
print("Document object     : ✅ imported")

# Token counter
import tiktoken
print(f"tiktoken            : {tiktoken.__version__}")

# PDF loader
import pymupdf
print(f"pymupdf             : {pymupdf.__version__}")

# CSV/Excel
import pandas as pd
print(f"pandas              : {pd.__version__}")

print()
print("✅ All libraries ready. You are good to go!")

Python version  : 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
Python path     : c:\Users\Lenovo\source\repos\RAG_Handson\Handson\.venv\Scripts\python.exe

langchain           : 1.2.12
langchain-community : 0.4.1
Text splitters      : ✅ all imported
Document object     : ✅ imported
tiktoken            : 0.12.0
pymupdf             : 1.27.2
pandas              : 3.0.1

✅ All libraries ready. You are good to go!


---
## Cell 2 — The Document Object

Before any chunking, you need to understand the core data structure in LangChain RAG.

Everything in LangChain RAG — whether loaded from a PDF, CSV, or plain text — gets wrapped into a `Document` object.

A `Document` has exactly two fields:

```
Document {
    page_content : str   ← the actual text that gets embedded and searched
    metadata     : dict  ← information ABOUT the text (source, page, section...)
}
```

Every chunk you create will be a `Document`. Every chunk you retrieve from the vector database will be a `Document`.

In [ ]:
# ── Create a Document manually ────────────────────────────────────────────────
# This is what every loader produces — understand the structure first

from langchain_core.documents import Document

doc = Document(
    page_content="Kubernetes is an open-source container orchestration platform "
                 "used to manage containerized applications at scale. "
                 "It was originally developed by Google.",
    metadata={
        "source": "kubernetes_intro.txt",
        "section": "Introduction",
        "page": 1,
        "doc_type": "technical"
    }
)

print("=== Document Structure ===")
print()
print(f"page_content : {doc.page_content}")
print()
print(f"metadata     : {doc.metadata}")
print()
print(f"Type         : {type(doc)}")

In [ ]:
# ── Access fields individually ────────────────────────────────────────────────

print("Accessing page_content:")
print(doc.page_content)
print()

print("Accessing individual metadata fields:")
print(f"  source   : {doc.metadata['source']}")
print(f"  section  : {doc.metadata['section']}")
print(f"  page     : {doc.metadata['page']}")
print(f"  doc_type : {doc.metadata['doc_type']}")

---
## Cell 3 — Sample Text for Practice

We will use this same text for all chunking experiments so you can directly compare results.

This text covers three clearly different topics on purpose — so you can see how well (or poorly) each splitter separates them.

In [ ]:
# ── Sample text for all experiments ──────────────────────────────────────────
# Three distinct topics in one document — watch how each splitter handles this

SAMPLE_TEXT = """
Kubernetes Overview

Kubernetes is an open-source container orchestration platform used to manage 
containerized applications at scale. It was originally developed by Google and 
is now maintained by the Cloud Native Computing Foundation.

Kubernetes automates deployment, scaling, and management of containerized 
applications. The core components include the API Server, Scheduler, Controller 
Manager, and etcd. The Scheduler assigns pods to nodes based on available 
resources and constraints.

Docker and Containers

Docker is a containerization platform that packages applications and their 
dependencies into portable containers. A container is a lightweight, standalone 
executable package that includes everything needed to run an application.

Docker containers share the host operating system kernel, making them much more 
efficient than traditional virtual machines. Docker Hub is the default registry 
for storing and distributing container images.

CI/CD Pipelines

Continuous Integration and Continuous Deployment pipelines automate the process 
of building, testing, and deploying software. CI ensures that code changes are 
automatically tested whenever they are committed to a repository.

Popular CI/CD tools include GitHub Actions, Jenkins, and GitLab CI. These tools 
integrate with version control systems and trigger automated workflows on events 
such as pull requests, merges, or scheduled intervals.
""".strip()

print(f"Total characters : {len(SAMPLE_TEXT)}")
print(f"Total words      : {len(SAMPLE_TEXT.split())}")
print()
print("Preview (first 200 chars):")
print(SAMPLE_TEXT[:200])

---
## Cell 4 — Fixed-Size Chunking

**What it does:** Splits text every N characters, regardless of where sentences or paragraphs end.

**Key parameters:**
- `chunk_size` — maximum characters per chunk
- `chunk_overlap` — characters shared between consecutive chunks

**Why we test this first:** It is the baseline. Every other method should produce better results than this.

In [ ]:
# ── Fixed-Size Chunking ───────────────────────────────────────────────────────

from langchain_text_splitters import CharacterTextSplitter

fixed_splitter = CharacterTextSplitter(
    separator="",        # no separator — pure character count
    chunk_size=300,      # max 300 characters per chunk
    chunk_overlap=50,    # 50 character overlap between chunks
)

fixed_chunks = fixed_splitter.split_text(SAMPLE_TEXT)

print(f"Total chunks created : {len(fixed_chunks)}")
print(f"Chunk size setting   : 300 characters")
print(f"Overlap setting      : 50 characters")
print()

# Inspect every chunk
for i, chunk in enumerate(fixed_chunks):
    print(f"{'─'*60}")
    print(f"CHUNK {i+1}  ({len(chunk)} characters)")
    print(f"{'─'*60}")
    print(chunk)
    print()

In [ ]:
# ── Observe the problems with fixed-size chunking ────────────────────────────
# Look at the first and last 50 characters of each chunk boundary

print("=== Chunk Boundary Analysis ===")
print("Last 60 chars of each chunk → First 60 chars of next chunk")
print()

for i in range(len(fixed_chunks) - 1):
    end_of_current   = fixed_chunks[i][-60:].replace('\n', '↵')
    start_of_next    = fixed_chunks[i+1][:60].replace('\n', '↵')
    print(f"Chunk {i+1} ends   : ...{end_of_current}")
    print(f"Chunk {i+2} starts : {start_of_next}...")
    print()

# 🔍 OBSERVATION: Notice where sentences get cut mid-way.
# This is exactly why fixed-size chunking is a baseline — not the final strategy.

---
## Cell 5 — Recursive Character Text Splitting

**What it does:** Tries to split at paragraph boundaries first. If a piece is still too large, tries sentence boundaries. If still too large, tries words.

**Separator priority (default):**
```
1st: "\n\n"  → paragraph breaks
2nd: "\n"    → line breaks
3rd: ". "    → sentence ends
4th: " "     → word boundaries
5th: ""      → character (last resort)
```

**This is the recommended default for 80% of RAG use cases.**

In [ ]:
# ── Recursive Character Text Splitting ───────────────────────────────────────

from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]  # priority order
)

recursive_chunks = recursive_splitter.split_text(SAMPLE_TEXT)

print(f"Total chunks created : {len(recursive_chunks)}")
print(f"Chunk size setting   : 300 characters")
print(f"Overlap setting      : 50 characters")
print()

for i, chunk in enumerate(recursive_chunks):
    print(f"{'─'*60}")
    print(f"CHUNK {i+1}  ({len(chunk)} characters)")
    print(f"{'─'*60}")
    print(chunk)
    print()

In [ ]:
# ── Boundary analysis for recursive chunks ───────────────────────────────────

print("=== Chunk Boundary Analysis (Recursive) ===")
print("Last 60 chars of each chunk → First 60 chars of next chunk")
print()

for i in range(len(recursive_chunks) - 1):
    end_of_current   = recursive_chunks[i][-60:].replace('\n', '↵')
    start_of_next    = recursive_chunks[i+1][:60].replace('\n', '↵')
    print(f"Chunk {i+1} ends   : ...{end_of_current}")
    print(f"Chunk {i+2} starts : {start_of_next}...")
    print()

# 🔍 OBSERVATION: Compare with fixed chunking boundaries above.
# Recursive splits at sentence/paragraph ends — much cleaner boundaries.

---
## Cell 6 — Side-by-Side Comparison: Fixed vs Recursive

Same text. Same chunk_size. Same overlap. Different result — because of the splitting strategy.

In [ ]:
# ── Side-by-side comparison ───────────────────────────────────────────────────

print("=" * 70)
print("COMPARISON: Fixed-Size  vs  Recursive Character Splitting")
print("Same text, same chunk_size=300, same overlap=50")
print("=" * 70)
print()

print(f"Fixed-Size   → {len(fixed_chunks)} chunks")
print(f"Recursive    → {len(recursive_chunks)} chunks")
print()

# Show chunk sizes
fixed_sizes     = [len(c) for c in fixed_chunks]
recursive_sizes = [len(c) for c in recursive_chunks]

print(f"Fixed-Size   chunk sizes : {fixed_sizes}")
print(f"Recursive    chunk sizes : {recursive_sizes}")
print()

print(f"Fixed    avg size : {sum(fixed_sizes)/len(fixed_sizes):.0f} chars")
print(f"Recursive avg size: {sum(recursive_sizes)/len(recursive_sizes):.0f} chars")
print()

# Show first chunk from each side by side
print("─" * 70)
print("CHUNK 1 — Fixed-Size:")
print(fixed_chunks[0])
print()
print("CHUNK 1 — Recursive:")
print(recursive_chunks[0])
print()

# 🔍 KEY QUESTION: Which Chunk 1 makes more sense when read alone?
# That is the chunk that will produce better embeddings and better retrieval.

---
## Cell 7 — Token Counting

**Why this matters:**  
Your `chunk_size` in LangChain is in **characters** — but embedding models have limits in **tokens**.

You need to verify your chunks are within the token limit of your embedding model.

| Embedding Model | Token Limit |
|---|---|
| `all-MiniLM-L6-v2` | 256 tokens |
| `BAAI/bge-small-en-v1.5` | 512 tokens |
| `text-embedding-3-small` (OpenAI) | 8191 tokens |

If your chunk exceeds the model limit → tokens are silently dropped → embedding is corrupted.

In [ ]:
# ── Count tokens per chunk ────────────────────────────────────────────────────
import tiktoken

# Using OpenAI's cl100k_base tokenizer (same as text-embedding-3-small)
# For sentence-transformers models the token count will be slightly different
# but this gives a very close approximation
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

print("=" * 60)
print("TOKEN COUNT PER CHUNK — Recursive Splitter")
print("=" * 60)
print()

MODEL_LIMIT = 256  # all-MiniLM-L6-v2 (the free local model we will use)

for i, chunk in enumerate(recursive_chunks):
    token_count = count_tokens(chunk)
    char_count  = len(chunk)
    status      = "✅ SAFE" if token_count <= MODEL_LIMIT else f"❌ EXCEEDS LIMIT ({MODEL_LIMIT})"
    print(f"Chunk {i+1:2d} : {char_count:4d} chars → {token_count:3d} tokens  {status}")

print()
all_tokens = [count_tokens(c) for c in recursive_chunks]
print(f"Max tokens in any chunk : {max(all_tokens)}")
print(f"Avg tokens per chunk    : {sum(all_tokens)/len(all_tokens):.1f}")
print(f"Model limit (MiniLM)    : {MODEL_LIMIT} tokens")

if max(all_tokens) <= MODEL_LIMIT:
    print()
    print(f"✅ All chunks are within the {MODEL_LIMIT}-token limit for all-MiniLM-L6-v2")
else:
    print()
    print(f"⚠️  Some chunks exceed the model limit — reduce chunk_size")

In [ ]:
# ── Token-Based Splitter (alternative to character-based) ─────────────────────
# This splits directly by token count — guarantees you never exceed the limit

from langchain_text_splitters import TokenTextSplitter

token_splitter = TokenTextSplitter(
    chunk_size=200,     # 200 tokens per chunk (well within 256 limit)
    chunk_overlap=20,   # 20 token overlap
)

token_chunks = token_splitter.split_text(SAMPLE_TEXT)

print(f"Token-based splitter created {len(token_chunks)} chunks")
print()
for i, chunk in enumerate(token_chunks):
    tokens = count_tokens(chunk)
    print(f"Chunk {i+1}: {tokens} tokens | {len(chunk)} chars")
    print(f"  Preview: {chunk[:80].replace(chr(10), ' ')}...")
    print()

---
## Cell 8 — Splitting Documents (not just text)

So far we used `.split_text()` which takes a plain string.  
In real RAG, you load files → get `Document` objects → then split them.

Use `.split_documents()` — it splits the content AND preserves the metadata.

In [ ]:
# ── Split Document objects (preserves metadata) ───────────────────────────────

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Simulate what a document loader returns
# (a PDF loader, text loader, etc. — all return a list of Documents)
source_documents = [
    Document(
        page_content=SAMPLE_TEXT,
        metadata={
            "source": "devops_guide.pdf",
            "page": 1,
            "doc_type": "technical",
            "author": "DevOps Team"
        }
    )
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# This is the real pattern — split_documents, not split_text
doc_chunks = splitter.split_documents(source_documents)

print(f"Input  : {len(source_documents)} document(s)")
print(f"Output : {len(doc_chunks)} chunk(s)")
print()

print("=== Inspecting chunks with metadata ===")
print()
for i, chunk in enumerate(doc_chunks):
    print(f"CHUNK {i+1}")
    print(f"  metadata     : {chunk.metadata}")
    print(f"  char count   : {len(chunk.page_content)}")
    print(f"  token count  : {count_tokens(chunk.page_content)}")
    print(f"  content      : {chunk.page_content[:100].replace(chr(10), ' ')}...")
    print()

---
## Cell 9 — Enriching Metadata

The metadata from the source document is carried into every chunk automatically.  
But you can also add MORE metadata to each chunk — like the chunk index, token count, or section title.

More metadata = better filtering and citation in retrieval.

In [ ]:
# ── Enriching chunk metadata ──────────────────────────────────────────────────

def enrich_chunk_metadata(chunks: list) -> list:
    """Add extra useful metadata to every chunk after splitting."""
    enriched = []
    for i, chunk in enumerate(chunks):
        # Add chunk-level fields
        chunk.metadata["chunk_index"]  = i
        chunk.metadata["total_chunks"] = len(chunks)
        chunk.metadata["char_count"]   = len(chunk.page_content)
        chunk.metadata["token_count"]  = count_tokens(chunk.page_content)
        enriched.append(chunk)
    return enriched


enriched_chunks = enrich_chunk_metadata(doc_chunks)

print("=== Enriched Chunk Metadata ===")
print()
for chunk in enriched_chunks:
    print(f"Chunk {chunk.metadata['chunk_index'] + 1} / {chunk.metadata['total_chunks']}")
    print(f"  source        : {chunk.metadata['source']}")
    print(f"  doc_type      : {chunk.metadata['doc_type']}")
    print(f"  chunk_index   : {chunk.metadata['chunk_index']}")
    print(f"  char_count    : {chunk.metadata['char_count']}")
    print(f"  token_count   : {chunk.metadata['token_count']}")
    print(f"  content       : {chunk.page_content[:80].replace(chr(10),' ')}...")
    print()

---
## Cell 10 — Experiment: Change Parameters and Observe

This cell is for **your own experiments**. Change the values and see what happens.

Questions to answer from your experiments:
1. What happens when you set `chunk_overlap=0`? Do any chunks lose context?
2. What happens when `chunk_size=100`? Are the chunks too small?
3. What happens when `chunk_size=1000`? Do multiple topics end up in one chunk?
4. Which chunk_size produces chunks where each one covers exactly one topic?

In [ ]:
# ── YOUR EXPERIMENT CELL ──────────────────────────────────────────────────────
# Change these values and re-run to see the effect

MY_CHUNK_SIZE    = 300   # ← try: 100, 200, 300, 500, 1000
MY_CHUNK_OVERLAP = 50    # ← try: 0, 20, 50, 100, 150

# ─────────────────────────────────────────────────────────────────────────────

exp_splitter = RecursiveCharacterTextSplitter(
    chunk_size=MY_CHUNK_SIZE,
    chunk_overlap=MY_CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

exp_chunks = exp_splitter.split_text(SAMPLE_TEXT)

print(f"Settings  : chunk_size={MY_CHUNK_SIZE}, overlap={MY_CHUNK_OVERLAP}")
print(f"Chunks    : {len(exp_chunks)}")
print(f"Avg chars : {sum(len(c) for c in exp_chunks) / len(exp_chunks):.0f}")
print(f"Avg tokens: {sum(count_tokens(c) for c in exp_chunks) / len(exp_chunks):.1f}")
print(f"Max tokens: {max(count_tokens(c) for c in exp_chunks)}")
print()

for i, chunk in enumerate(exp_chunks):
    print(f"[Chunk {i+1}] {count_tokens(chunk)} tokens | {len(chunk)} chars")
    print(chunk)
    print()

---
## Cell 11 — Summary and What's Next

### What you practiced in this notebook

| Concept | What You Did |
|---|---|
| `Document` object | Created one manually, accessed `page_content` and `metadata` |
| Fixed-size chunking | Split text by character count, observed broken boundaries |
| Recursive chunking | Split by paragraph → sentence → word priority, compared with fixed |
| Token counting | Counted tokens per chunk, verified against model limits |
| `split_documents()` | Split `Document` objects while preserving metadata |
| Metadata enrichment | Added chunk index, token count, and char count to metadata |
| Experiments | Changed chunk_size and overlap to observe effects |

---

### Key things to remember

1. **`chunk_size` in LangChain = characters, NOT tokens** — always verify token count separately
2. **Recursive is better than Fixed** for almost all text — it respects sentence boundaries
3. **Overlap is not optional** — without it, context is lost at every chunk boundary
4. **Always check max tokens** against your embedding model's limit before indexing
5. **Metadata is as important as content** — it enables filtering, citation, and debugging

---

### Next notebook

`02_loading_documents.ipynb` — Load real files: PDF, CSV, Word document, JSON, Markdown.  
Apply the same chunking strategies to real documents and observe how different sources behave.

In [ ]:
# ── Final summary printout ────────────────────────────────────────────────────

print("=" * 60)
print("NOTEBOOK SUMMARY")
print("=" * 60)
print()
print(f"Sample text         : {len(SAMPLE_TEXT)} chars, {len(SAMPLE_TEXT.split())} words")
print()
print("Chunking Results (chunk_size=300, overlap=50):")
print(f"  Fixed-size chunks  : {len(fixed_chunks)}")
print(f"  Recursive chunks   : {len(recursive_chunks)}")
print(f"  Token-based chunks : {len(token_chunks)}")
print()
print("Token counts (recursive, MiniLM limit = 256 tokens):")
rec_tokens = [count_tokens(c) for c in recursive_chunks]
print(f"  Min tokens per chunk : {min(rec_tokens)}")
print(f"  Max tokens per chunk : {max(rec_tokens)}")
print(f"  Avg tokens per chunk : {sum(rec_tokens)/len(rec_tokens):.1f}")
safe = all(t <= 256 for t in rec_tokens)
print(f"  All within 256 limit : {'✅ Yes' if safe else '❌ No — reduce chunk_size'}")
print()
print("✅ Notebook complete. You are ready for the next step.")